**Group Name:** _____  
**Members:** _____

# CS5489 — Test Code Template (2026)

**Purpose:** Verify that the submitted `model.pth` can be loaded and used to predict RLE masks from RGB test images. **No ground-truth labels are required.** Students must complete the marked sections so TAs can run the notebook by changing only the paths below.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import sys
sys.path.insert(0, str(Path('../baseline').resolve()))
from project import SmallUNet
from sliding import predict_binary_mask as predict_sliding_mask

# TA should only need to change these paths.
test_dir = Path("data/test_images")
model_path = Path("B/model.pth")
output_csv = Path("submission.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Student implementation

**Students:** Complete the next three functions. Include all model architecture definitions and dependencies in the submitted notebook (or accompanying files). Your preprocessing must match training. `predict_binary_mask` must return a 2D binary array of shape `(original_height, original_width)`. Do not use test labels.

In [ ]:
def build_model():
    return SmallUNet()


def load_model(model_path, device):
    model = build_model().to(device)
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    model.image_size = checkpoint['image_size']
    model.eval()
    return model


def predict_binary_mask(model, image, device):
    return predict_sliding_mask(model, image, device)

## Shared RLE encoder (do not change)

Uses **column-major** flattening, **1-based** starting positions, and `start length` pairs. Empty mask → empty string.

In [ ]:
def mask_to_rle(mask):
    mask = np.asarray(mask)
    if mask.ndim != 2 or not np.isin(mask, [0, 1]).all():
        raise ValueError("Predicted mask must be 2D and contain only 0/1")
    pixels = mask.astype(np.uint8).ravel(order="F")
    padded = np.pad(pixels, (1, 1), constant_values=0)
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    starts = changes[::2] + 1
    lengths = changes[1::2] - changes[::2]
    return " ".join(str(v) for pair in zip(starts, lengths) for v in pair)

assert mask_to_rle(np.array([[0, 0, 1, 1, 1], [0, 0, 0, 0, 1]], dtype=np.uint8)) == "5 1 7 1 9 2"
print("RLE encoder check passed")

## Inference verification and CSV export

This cell must run **without access to any test ground-truth labels**. It checks every predicted mask shape and writes the TA template CSV with `ID,Predicted`. IDs are image filename stems and retain leading zeros.

In [ ]:
def run_test(test_dir, model_path, output_csv, device):
    if not test_dir.is_dir():
        raise FileNotFoundError(f"Test image directory not found: {test_dir}")
    if not model_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {model_path}")

    extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
    images = sorted(p for p in test_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in extensions)
    if not images:
        raise ValueError(f"No images found in {test_dir}")
    ids = [p.stem for p in images]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate image IDs across file extensions")

    model = load_model(model_path, device)
    rows = []
    for index, path in enumerate(images, 1):
        with Image.open(path) as im:
            rgb = im.convert("RGB")
            height, width = rgb.height, rgb.width
            with torch.inference_mode():
                mask = np.asarray(predict_binary_mask(model, rgb, device))
        if mask.shape != (height, width):
            raise ValueError(f"{path.name}: expected {(height, width)}, got {mask.shape}")
        rle = mask_to_rle(mask)
        rows.append({"ID": path.stem, "Predicted": rle})
        if index == 1 or index % 100 == 0 or index == len(images):
            print(f"Processed {index}/{len(images)} images")

    df = pd.DataFrame(rows, columns=["ID", "Predicted"])
    df.to_csv(output_csv, index=False)
    print(f"PASS: checkpoint loaded and {len(df)} masks predicted")
    print(f"Saved: {output_csv}")
    print(df.assign(Predicted=df.Predicted.str.slice(0, 70)).head())
    return df

submission = run_test(test_dir, model_path, output_csv, device)

## Submission checklist

- [ ] Run all cells successfully with the submitted `model.pth`.
- [ ] Model loads in evaluation mode without training data or hidden labels.
- [ ] Every test image produces a full-resolution 0/1 mask.
- [ ] `submission.csv` contains one row per test image, with columns `ID,Predicted`.
- [ ] Submit the runnable `test.ipynb`, checkpoint, and any required local model code/dependencies as instructed by the course.

**For TAs:** Change `test_dir` and `model_path` in the configuration cell, then Run All. The notebook verifies inference and exports the predicted masks for evaluation.